# Held-out-edge modularity — de-circularizing scProto's headline metric, generalized

**Reviewer concern this answers** (F5RB, e9Ho — full rationale in `../rebuttal_plan.md`):

> "The community loss directly encourages prototype assignment similarity to match the input affinity graph. Later, modularity is computed using the per-batch affinity subgraph... high modularity is partly expected." — e9Ho

> "Modularity is closely related to the training affinity graph. It is also used for early stopping and then reported as a main result." — F5RB

**What this notebook actually tests.** Not "does scProto have high modularity" (that's the circular headline number) but **can scProto's encoder place cells it never saw an edge for into the correct community, from gene expression alone, or does it just memorize the specific edges it was fed?** Concretely: hide 20% of the affinity graph's edges from training entirely, train scProto only on the visible 80%, then score modularity using *only* the hidden 20% — edges the model never had a training signal for.

**Rows in the comparison, and what each one is for:**

1. **scProto (masked-trained) vs. SEACells (identical masked graph)** — the direct de-circularization test both reviewers asked for. *Caveat: a win here is confounded with batch-correction* — SEACells has no batch-effect handling, scProto does (via Stage-1 scPoli pretraining) — so this alone doesn't separate "generalizes better" from "corrects batch effects better."
2. **scProto vs. scPoli-Stage1-latent + {SEACells, Leiden}** — the comparison that actually isolates the claim we want to make. Both start from the *same* scPoli Stage-1 checkpoint (identical batch-correction backbone), so any gap here is attributable to Stage-2's affinity-supervised training, not batch handling. scPoli-Stage1 itself never trains on any affinity graph — plain conditional VAE, recon + KL only — so it's the natural "no affinity supervision at all" control.
3. **scProto vs. scVI-latent / Harmony-latent + {SEACells, Leiden}** (bonus rows) — same "no affinity supervision" logic, different batch-correction mechanisms, reused free from `batch_correct_then_cluster_baselines.ipynb` (E1).

Rows 2/3 cost nothing new to compute — none of {scPoli-Stage1, scVI, Harmony} ever consume the PCA-arbf affinity graph in any form, masked or not, so their existing E1 assignments are simply re-scored against the new held-out test edges. Row 1 is the only pair that requires retraining/refitting on the masked graph.

**Masking stays uniform/random**, not targeted at rare cell types — deliberately hiding *more* rare-cell edges would invite a "you engineered the test set" objection. Instead, the held-out set is split into two slices *after* random masking: edges touching a rare cell type (bottom-quartile frequency, `mc_metric_utils.get_rare`, same definition used everywhere else in this rebuttal) vs. edges between two common cell types. This targets the hypothesis that batch-correction VAEs' reconstruction+KL objective is dominated by common patterns and may blur rare states (`rebuttal_plan.md`'s "Mechanistic answer" section) without cherry-picking what got hidden.

**Decisions carried over from the single-dataset version of this notebook, unchanged:**
- 20% of edges held out, symmetric, per-cell floor of ~5 remaining visible neighbors so masking doesn't starve any cell's visible graph.
- scProto's negative sampler is *not* patched to exclude held-out edges — leaving Nassoc's negative pressure active on masked pairs keeps scProto and SEACells under the same masking discipline; if scProto still recovers held-out structure despite that pressure, the result is conservative, not inflated.
- SEACells' native waypoint init reads `ad.obsm['X_pca']` directly (not the masked kernel) for archetype seeding — a known, minor asymmetry (init only affects the starting point of an iterative fit, and it's the same behavior SEACells' own published PCA baseline already has). Deliberately not fixed — swapping to `compute_seacells_own_affinity` (seeds also from the masked graph) was considered and dropped as unnecessary complexity for a second-order effect.
- Early stopping during the masked-graph scProto run still uses modularity on the *masked/visible* graph only, never the held-out edges — scoring early stopping against the held-out set would reintroduce the same double-dipping problem one level down.
- Leiden directly on the masked PCA-arbf graph is **not** included as a row — it's the same batch-uncorrected graph SEACells already gets scored on, so it repeats SEACells' batch-effect handicap with a different clustering algorithm rather than adding a new axis of comparison.

**Scope: all 3 RNA-seq datasets** (pancreas, lung, pbmc-immune) — matching `batch_correct_then_cluster_baselines.ipynb` (E1), which is what makes reusing its scPoli/scVI/Harmony assignments possible. The single-dataset `s28nsc` proof-of-concept this notebook used to run is superseded by this version — s28nsc has no E1 baselines to reuse, and the reviewers' circularity complaint was raised about the scIB datasets specifically, not the spatial one.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 141.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 136.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 711.2/711.2 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.3/102.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 116.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 31.1 MB/s eta 0:00:00
 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 132.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.66.0 which is incompatible.
tsfresh 0.21.2 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 1.13.1 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you hav

In [1]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# numpy/scipy/anndata are C-extension linked, an in-process upgrade alone won't
# reliably take effect on already-imported modules.

In [2]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real
# test.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'umap-learn': 'umap', 'harmonypy': 'harmonypy',
    'faiss-cpu': 'faiss',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")

  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)


/tmp/ipykernel_5468/4100688168.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  ver = getattr(mod, '__version__', '?')


  OK   anndata          (import anndata, version 0.13.2)


/tmp/ipykernel_5468/4100688168.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  ver = getattr(mod, '__version__', '?')


  OK   scanpy           (import scanpy, version 1.12.3)


  FAIL scarches         (import scarches): ImportError: cannot import name 'read' from 'anndata' (/usr/local/lib/python3.12/dist-packages/anndata/__init__.py)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.5.10)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   umap-learn       (import umap, version 0.5.12)
  OK   harmonypy        (import harmonypy, version 2.0.0)
  OK   faiss-cpu        (import faiss, version 1.14.1)

1 package(s) failed to import: ['scarches'] -- re-run that package's specific pip install line above and check its full error output before proceeding.


In [3]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py

nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [4]:
# Extra imports not already covered by nb_setup.py (which already pulls in
# run_mc_task, show_table, clean_run_names, etc. via
# `from interpretable_ssl.evaluation.paper_figures import *`).
import os
import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pickle
import anndata
import scanpy as sc

import SEACells
import SEACells.core
import SEACells.build_graph

from interpretable_ssl.experiments.tasks import run_mc_task, LAMBDA_PROTO_UMAP_PRECON
from interpretable_ssl.evaluation.mc_metric_utils import calc_purity, get_rare, calc_modularity_per_batch
from interpretable_ssl.augmenters.graph_generator import save_affinity
from interpretable_ssl.configs.paths import get_affinity_path, get_dataset_model_dir, get_seacell_model_dir
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.evaluation.metric_helpers.metacell_metrics import compute_seacells_from_affinity
from interpretable_ssl.evaluation.metric_helpers.embedding_metrics import load_seacell

print("extra imports ready")

# SEACells (installed unpinned from GitHub main, --no-deps) calls AnnData(..., dtype=...)
# internally in a few places -- removed kwarg in the anndata version this install
# resolves to. Patch: drop a stray dtype= kwarg instead of raising.
_orig_anndata_init = anndata.AnnData.__init__
def _patched_anndata_init(self, *args, **kwargs):
    kwargs.pop('dtype', None)
    _orig_anndata_init(self, *args, **kwargs)
anndata.AnnData.__init__ = _patched_anndata_init
print("Patched AnnData.__init__ to tolerate a stray dtype= kwarg (SEACells compatibility)")

extra imports ready
Patched AnnData.__init__ to tolerate a stray dtype= kwarg (SEACells compatibility)


## Config

In [5]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']
AFFINITY = 'arbf'

FRAC_HELDOUT = 0.20          # fraction of edges hidden from training
MIN_VISIBLE_DEGREE = 5       # floor on remaining visible neighbors per cell after masking
SEED = 0
RARE_QUANTILE = 0.25         # matches mc_metric_utils.get_rare's default

HELDOUT_TAG = f'{AFFINITY}_heldout{int(FRAC_HELDOUT * 100)}_seed{SEED}'
TESTEDGES_TAG = f'{HELDOUT_TAG}_testedges'

GRAPH_DIR = os.path.join(os.environ['CODE_DIR'], 'graphs')
os.makedirs(GRAPH_DIR, exist_ok=True)

CVAE_EPOCHS = 50
BATCH_SIZE = 1024

def common_kwargs():
    return dict(
        cvae_epochs=CVAE_EPOCHS,
        train_epochs=50,
        eval_freq=3,
        patience=6,
        batch_size=BATCH_SIZE,
        umap_steps_per_epoch=500,
        lambda_config=LAMBDA_PROTO_UMAP_PRECON | {'nassoc_agg': 'max'},
    )

# Baseline (unmasked 'arbf') run -- reloads the existing canonical run, never retrains it here.
LOAD_BASELINE = True

# Held-out-edge run -- the new thing this notebook produces per dataset. False trains
# fresh; flip to True on a re-run (after a first successful run) to just reload.
LOAD_HELDOUT = True

# Two-step baselines reused from batch_correct_then_cluster_baselines.ipynb (E1) -- none
# of these train on any affinity graph, so nothing here needs remasking or retraining;
# we just re-score their already-saved cluster assignments against the held-out test
# edges. Folder-tag convention matches E1's CORRECTION_METHODS. 'stage1z' (scPoli) is
# the primary secondary comparison (see intro cell); scvi/harmony are free bonus rows.
TWO_STEP_METHODS = ['stage1z', 'scvi', 'harmony']
METHOD_DISPLAY_NAMES = {
    'stage1z': 'scPoli (Stage-1)',
    'scvi': 'scVI',
    'harmony': 'Harmony',
}

HELDOUT_TAG, TESTEDGES_TAG

('arbf_heldout20_seed0', 'arbf_heldout20_seed0_testedges')

## Helper functions

In [6]:
def split_affinity_edges(aff, frac_heldout=0.2, min_visible_degree=5, seed=0):
    """Split a symmetric sparse affinity matrix into visible (train) and held-out (test) edges.

    Not promoted into graph_generator.py -- this notebook is the only place a held-out-edge
    split like this is needed so far; move it there if a second experiment reuses it.
    """
    A = sp.csr_matrix(aff)
    A = (A + A.T) / 2
    A = A.tocoo()

    upper = A.row < A.col
    rows, cols, vals = A.row[upper], A.col[upper], A.data[upper]
    n_edges = len(rows)

    rng = np.random.default_rng(seed)
    order = rng.permutation(n_edges)

    degree_count = np.array((A.tocsr() > 0).sum(axis=1)).ravel()
    remaining_degree = degree_count.copy()

    held_out = np.zeros(n_edges, dtype=bool)
    n_target = int(round(frac_heldout * n_edges))

    for idx in order:
        if held_out.sum() >= n_target:
            break
        i, j = rows[idx], cols[idx]
        if remaining_degree[i] - 1 < min_visible_degree or remaining_degree[j] - 1 < min_visible_degree:
            continue
        held_out[idx] = True
        remaining_degree[i] -= 1
        remaining_degree[j] -= 1

    def _symmetrize(r, c, v, shape):
        rr = np.concatenate([r, c])
        cc = np.concatenate([c, r])
        vv = np.concatenate([v, v])
        return sp.csr_matrix((vv, (rr, cc)), shape=shape)

    aff_train = _symmetrize(rows[~held_out], cols[~held_out], vals[~held_out], A.shape)
    aff_test = _symmetrize(rows[held_out], cols[held_out], vals[held_out], A.shape)

    actual_frac = held_out.mean()
    print(f'held out {held_out.sum()}/{n_edges} undirected edges '
          f'({actual_frac:.1%}, target was {frac_heldout:.0%})')
    return aff_train, aff_test

In [7]:
def modularity_on_edges(A, assignments):
    """Weighted Newman modularity of `assignments` scored against adjacency `A`.

    Q = (1/2m) * sum_k [ e_k - d_k^2 / (2m) ]
    Identical formula to Trainer.modularity() / mc_metric_utils.compute_modularity(),
    parameterized on an explicit A instead of the model's own training graph.
    """
    A = sp.csr_matrix(A)
    A = (A + A.T) / 2
    degrees = np.array(A.sum(axis=1)).ravel()
    two_m = degrees.sum()
    if two_m == 0:
        return 0.0
    Q = 0.0
    for c in np.unique(assignments):
        mask = (assignments == c)
        e_k = A[mask][:, mask].sum()
        d_k = degrees[mask].sum()
        Q += (e_k - d_k * d_k / two_m) / two_m
    return float(Q)

In [8]:
def held_out_same_cluster_rate(aff, assignments, cell_mask=None, require='any'):
    """Fraction of held-out edges with both endpoints in the same cluster/prototype.

    Complements modularity_on_edges() with a metric that stays interpretable on a
    small, sparse edge subset (e.g. rare-cell-touching edges only) -- full Newman
    modularity's null-model term uses global degree, which gets noisy on a tiny
    restricted subgraph. Reports both the plain edge-count rate and an
    affinity-weighted rate.

    Args:
        aff:          scipy sparse adjacency (the held-out test edges, or a subset).
        assignments:  (N,) array, cluster/prototype id per cell, aligned to aff's rows.
        cell_mask:    optional (N,) boolean array (e.g. a rare-cell-type mask).
        require:      'any'  -- keep edges where at least one endpoint is masked-True
                                 (use with a rare-type mask: "touches a rare cell").
                      'all'  -- keep edges where both endpoints are masked-True
                                 (use with a common-type mask: "both cells common").

    Returns: dict with 'rate', 'weighted_rate', 'n_edges'.
    """
    A = sp.csr_matrix(aff)
    A = (A + A.T) / 2
    A = A.tocoo()
    upper = A.row < A.col
    rows, cols, vals = A.row[upper], A.col[upper], A.data[upper]

    if cell_mask is not None:
        cell_mask = np.asarray(cell_mask)
        if require == 'any':
            edge_mask = cell_mask[rows] | cell_mask[cols]
        elif require == 'all':
            edge_mask = cell_mask[rows] & cell_mask[cols]
        else:
            raise ValueError(f"require must be 'any' or 'all', got {require!r}")
        rows, cols, vals = rows[edge_mask], cols[edge_mask], vals[edge_mask]

    if len(rows) == 0:
        return {'rate': None, 'weighted_rate': None, 'n_edges': 0}

    same = assignments[rows] == assignments[cols]
    weighted_rate = float((same * vals).sum() / vals.sum()) if vals.sum() > 0 else None
    return {'rate': float(same.mean()), 'weighted_rate': weighted_rate, 'n_edges': int(len(rows))}

In [9]:
def _find_existing_leiden_dir(ds_id, tag):
    """leiden's save folder name includes n_clusters (unknown ahead of time) --
    find it by prefix match, same convention batch_correct_then_cluster_baselines.ipynb uses.
    """
    base_dir = get_dataset_model_dir(ds_id)
    if not os.path.isdir(base_dir):
        return None
    for entry in sorted(os.listdir(base_dir)):
        if entry.startswith(f'leiden_{tag}_K') and os.path.exists(os.path.join(base_dir, entry, 'metrics.json')):
            return os.path.join(base_dir, entry)
    return None


def load_two_step_assignments(ds_id, method, cell_index):
    """Load `method`'s already-saved SEACells + Leiden cluster assignments from E1
    (batch_correct_then_cluster_baselines.ipynb), reindexed to `cell_index` (the E2
    run's own ad.obs_names order) so they line up with aff_test's rows.

    None of these methods (stage1z / scvi / harmony) train on any affinity graph --
    masked or not -- so nothing here is retrained; this purely re-scores existing
    assignments against a new test set.

    Reads cell_assignments.csv directly (written by _save_seacell_umap_data /
    save_baseline_umap_data for SEACells and Leiden respectively) instead of going
    through load_seacell()'s h5ad base+delta reconstruction -- that path raised
    FileNotFoundError for seacell_X_stage1z on all 3 datasets even though the run
    clearly completed (its numbers show up in batch_correct_then_cluster_baselines.ipynb's
    own rare_celltype_purity_table, which reads via this same csv). Reading the csv
    directly sidesteps whatever broke the h5ad reconstruction rather than debugging it.

    A method missing from disk (E1 not yet run for it on this dataset) prints a
    warning and returns None for that slot rather than crashing the whole comparison.
    """
    tag = f'X_{method}'
    out = {'seacells': None, 'leiden': None}

    seacell_dir = get_seacell_model_dir(ds_id, tag)
    seacell_csv = os.path.join(seacell_dir, 'cell_assignments.csv')
    if os.path.exists(seacell_csv):
        df = pd.read_csv(seacell_csv).set_index('cell_id')
        s = df['metacell_id'].reindex(cell_index)
        if s.isna().any():
            print(f"[{ds_id}] WARNING: seacell_{tag} assignments missing "
                  f"{int(s.isna().sum())} cells after reindex -- skipping this row.")
        else:
            out['seacells'] = s.values.astype(int)
    else:
        print(f"[{ds_id}] seacell_{tag} cell_assignments.csv not found at {seacell_csv} -- skipping.")

    leiden_dir = _find_existing_leiden_dir(ds_id, tag)
    if leiden_dir is not None:
        df = pd.read_csv(os.path.join(leiden_dir, 'cell_assignments.csv')).set_index('cell_id')
        s = df['metacell_id'].reindex(cell_index)
        if s.isna().any():
            print(f"[{ds_id}] WARNING: leiden_{tag} assignments missing "
                  f"{int(s.isna().sum())} cells after reindex -- skipping this row.")
        else:
            out['leiden'] = s.values.astype(int)
    else:
        print(f"[{ds_id}] leiden_{tag} not found on disk yet -- skipping.")

    return out

## Run one dataset

`run_heldout_experiment(ds_id)` does the full pipeline for one dataset: load/mask the affinity
graph, train scProto + fit SEACells on the masked graph, reuse E1's scPoli/scVI/Harmony
assignments, and score every method's final partition against the same held-out test edges
(overall, and split into rare-cell-touching vs. common-only slices).

In [10]:
def run_heldout_experiment(ds_id):
    lk = DATASETS[ds_id]['label_key']
    bk = DATASETS[ds_id].get('batch_key')
    kwargs = common_kwargs()

    # --- 1. Load the full affinity graph (reuses the existing canonical run) ---
    t_base, res_base, _ = run_mc_task(ds_id, affinity_type=AFFINITY, load_umap=LOAD_BASELINE, **kwargs)
    ad = t_base.train_ds.adata
    aff_full = t_base.train_ds.aff_raw if hasattr(t_base.train_ds, 'aff_raw') else t_base.train_ds.aff
    aff_full = sp.csr_matrix(aff_full)
    n_cells = len(ad)
    print(f"[{ds_id}] {n_cells} cells, {aff_full.nnz} nonzero affinity entries")

    # --- 2. Mask edges: 20% held out, symmetric, per-cell floor, seeded ---
    # Load the previously-saved split if it exists, instead of recomputing.
    # split_affinity_edges is deterministic given the same aff_full + SEED, but
    # loading the exact file scProto/SEACells were actually fit against removes any
    # dependence on that assumption -- if aff_full construction ever changes upstream
    # (different PCA/kNN run, etc.), a silent recompute could produce a different
    # split than the one a *reloaded* scProto checkpoint was actually trained on,
    # and we'd be scoring against edges that were partly visible during training.
    test_path = get_affinity_path(ds_id, n_cells, affinity_type=TESTEDGES_TAG, graph_dir=GRAPH_DIR)
    train_path = get_affinity_path(ds_id, n_cells, affinity_type=HELDOUT_TAG, graph_dir=GRAPH_DIR)
    if os.path.exists(test_path) and os.path.exists(train_path):
        with open(test_path, 'rb') as f:
            aff_test = pickle.load(f)
        with open(train_path, 'rb') as f:
            aff_train = pickle.load(f)
        print(f"[{ds_id}] loaded existing held-out split from disk (not recomputed): {test_path}")
    else:
        aff_train, aff_test = split_affinity_edges(
            aff_full, frac_heldout=FRAC_HELDOUT, min_visible_degree=MIN_VISIBLE_DEGREE, seed=SEED,
        )
        save_affinity(aff_train, ds_id, n_cells, HELDOUT_TAG, graph_dir=GRAPH_DIR)
        with open(test_path, 'wb') as f:
            pickle.dump(aff_test, f)
        print(f"[{ds_id}] saved held-out test edges: {test_path}")

    # --- 3. Train scProto on the masked (visible-edge) graph ---
    t_ho, res_ho, _ = run_mc_task(ds_id, affinity_type=HELDOUT_TAG, load_umap=LOAD_HELDOUT, **kwargs)
    print(f"[{ds_id}] held-out-edge run (train-graph modularity, not the headline number): {res_ho}")

    # --- 4. Run SEACells on the same masked graph ---
    # Cached as a small cell_assignments.csv only (cell_id + integer metacell id) --
    # deliberately NOT an h5ad: the per-cell h5ad files SEACells normally writes are
    # large (the pbmc-immune base alone was ~95MB) and are exactly what went missing
    # from disk for the scPoli/scVI/Harmony runs, presumably cleaned up for space at
    # some point. A skip-if-exists check here also means a rerun neither re-fits
    # (compute_seacells_from_affinity has no caching of its own) nor re-saves.
    n_seacells = t_ho.nmb_prototypes
    seacell_masked_dir = get_seacell_model_dir(ds_id, HELDOUT_TAG)
    seacell_masked_csv = os.path.join(seacell_masked_dir, 'cell_assignments.csv')
    if os.path.exists(seacell_masked_csv):
        assignments_seacell = (
            pd.read_csv(seacell_masked_csv).set_index('cell_id')['metacell_id']
            .reindex(ad.obs_names).values
        )
        print(f"[{ds_id}] loaded existing SEACells-on-masked-graph assignments (not refit): {seacell_masked_csv}")
    else:
        ad_sc, SEACell_ad, seacell_model = compute_seacells_from_affinity(
            ad, n_SEACells=n_seacells, ds_name=ds_id, affinity_type=HELDOUT_TAG,
            n_components=50, k_neighbors=50, graph_dir=GRAPH_DIR, build_kernel_on='X_pca',
        )
        assignments_seacell = pd.factorize(ad_sc.obs['SEACell'])[0]
        os.makedirs(seacell_masked_dir, exist_ok=True)
        pd.DataFrame({'cell_id': ad.obs_names, 'metacell_id': assignments_seacell}).to_csv(
            seacell_masked_csv, index=False,
        )
        print(f"[{ds_id}] saved SEACells-on-masked-graph assignments (csv only, no h5ad): {seacell_masked_csv}")
    print(f"[{ds_id}] SEACells: {len(np.unique(assignments_seacell))} non-empty metacells "
          f"(of {n_seacells} requested)")

    # --- 5. Rare-cell-type mask (post-hoc stratification, same held-out edge set) ---
    rare_types = get_rare(ad, lk, thr=RARE_QUANTILE)
    rare_mask = ad.obs[lk].isin(rare_types).values
    print(f"[{ds_id}] {int(rare_mask.sum())}/{n_cells} cells are rare-type "
          f"({len(rare_types)} rare types of {ad.obs[lk].nunique()})")

    # --- 6. Score every method's final assignment against the SAME held-out test edges ---
    assignments_scproto, _ = t_ho._get_assignments()
    assert len(assignments_scproto) == n_cells, \
        f"scProto assignment length {len(assignments_scproto)} != {n_cells} cells in ad " \
        "-- cell ordering between t_ho and ad may not match, do not trust these numbers yet."

    rows = {
        'scProto (masked-trained)': np.asarray(assignments_scproto),
        'SEACells (same masked graph)': np.asarray(assignments_seacell),
    }

    for method in TWO_STEP_METHODS:
        disp = METHOD_DISPLAY_NAMES[method]
        loaded = load_two_step_assignments(ds_id, method, ad.obs_names)
        if loaded['seacells'] is not None:
            rows[f'SEACells ({disp})'] = loaded['seacells']
        if loaded['leiden'] is not None:
            rows[f'Leiden ({disp})'] = loaded['leiden']

    # Per-batch held-out modularity: mean/std across batches, using ONLY the held-out
    # test edges (aff_test) -- reuses calc_modularity_per_batch (same function/formula
    # the paper's own headline modularity uses), just parameterized on the held-out
    # graph instead of the training graph. This is the variance source requested for
    # this table -- matches Appendix F's own "mean and std across batches" definition.
    # Purely a recomputation over already-saved aff_test + assignments; no retraining.
    records = []
    for name, assign in rows.items():
        overall = held_out_same_cluster_rate(aff_test, assign)
        rare = held_out_same_cluster_rate(aff_test, assign, cell_mask=rare_mask, require='any')
        common = held_out_same_cluster_rate(aff_test, assign, cell_mask=~rare_mask, require='all')

        held_out_mod_batch_mean = held_out_mod_batch_std = None
        if bk is not None and bk in ad.obs.columns:
            batch_mod_s = calc_modularity_per_batch(aff_test, np.asarray(assign), ad.obs[bk].values)
            held_out_mod_batch_mean = float(batch_mod_s.mean())
            held_out_mod_batch_std = float(batch_mod_s.std())

        records.append({
            'dataset': ds_id,
            'method': name,
            'held_out_modularity': modularity_on_edges(aff_test, assign),
            'held_out_modularity_batch_mean': held_out_mod_batch_mean,
            'held_out_modularity_batch_std': held_out_mod_batch_std,
            'held_out_same_cluster_rate': overall['rate'],
            'held_out_same_cluster_rate_weighted': overall['weighted_rate'],
            'rare_edge_same_cluster_rate': rare['rate'],
            'rare_edge_n': rare['n_edges'],
            'common_edge_same_cluster_rate': common['rate'],
        })

    return pd.DataFrame.from_records(records)

## Run all 3 datasets

In [ ]:
all_results = []
for ds_id in RNA_SEQ_DATASETS:
    print(f"\n=== {ds_id} ===")
    all_results.append(run_heldout_experiment(ds_id))

df_heldout = pd.concat(all_results, ignore_index=True)
df_heldout


=== pancreas ===


 captum (see https://github.com/pytorch/captum).


dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}
📊 EdgeDataset: 1155146 edges
   Weight range: [0.0320, 0.9688]
   umap_steps_per_epoch=500 → 512000 edges/epoch (of 1155146 total

  0%|          | 0/16 [00:00<?, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

...

Selecting 60 cells from greedy initialization.
Randomly initialized A matrix.
Setting convergence threshold at 0.46771
Starting iteration 1.
Completed iteration 1.
Converged after 9 iterations.
Starting iteration 10.
Completed iteration 10.
Converged after 10 iterations.


100%|██████████| 300/300 [00:02<00:00, 101.84it/s]


[pbmc-immune] saved SEACells-on-masked-graph assignments (csv only, no h5ad): /content/drive/MyDrive/models/pbmc-immune/seacell_arbf_heldout20_seed0/cell_assignments.csv
[pbmc-immune] SEACells: 300 non-empty metacells (of 300 requested)
[pbmc-immune] 1034/33506 cells are rare-type (4 rare types of 16)


  0%|          | 0/33 [00:00<?, ?it/s]

,dataset,method,held_out_modularity,held_out_modularity_batch_mean,held_out_modularity_batch_std,held_out_same_cluster_rate,held_out_same_cluster_rate_weighted,rare_edge_same_cluster_rate,rare_edge_n,common_edge_same_cluster_rate
0,pancreas,scProto (masked-trained),0.669509,0.591442,0.091741,0.685374,0.704154,0.307529,943,0.688484
1,pancreas,SEACells (same masked graph),0.556038,0.473448,0.077725,0.536848,0.563310,0.454931,943,0.537522
2,pancreas,SEACells (scPoli (Stage-1)),0.273845,0.251897,0.064553,0.260771,0.279859,0.270414,943,0.260692
3,pancreas,Leiden (scPoli (Stage-1)),0.386216,0.356448,0.057773,0.384521,0.400284,0.389183,943,0.384483
4,pancreas,SEACells (scVI),0.311387,0.273860,0.066151,0.300281,0.318887,0.213150,943,0.300998
5,pancreas,Leiden (scVI),0.363387,0.341359,0.114691,0.354422,0.369874,0.497349,943,0.353245
6,pancreas,SEACells (Harmony),0.259259,0.236082,0.071026,0.244418,0.266100,0.246023,943,0.244405
7,pancreas,Leiden (Harmony),0.569431,0.528314,0.091180,0.600632,0.613755,0.534464,943,0.601177
8,lung,scProto (masked-trained),0.724598,0.661286,0.023887,0.735616,0.750599,0.725261,10075,0.736061
9,lung,SEACells (same masked graph),0.597554,0.521030,0.070371,0.579427,0.602647,0.625211,10075,0.577461


In [ ]:
pd.set_option('display.width', 160)
summary = df_heldout.set_index(['dataset', 'method'])[
    ['held_out_modularity', 'held_out_modularity_batch_mean', 'held_out_modularity_batch_std',
     'held_out_same_cluster_rate', 'rare_edge_same_cluster_rate',
     'common_edge_same_cluster_rate', 'rare_edge_n']
].round(3)
summary

held_out_modularity  held_out_modularity_batch_mean  held_out_modularity_batch_std  held_out_same_cluster_rate  \
dataset     method                                                                                                                                         
pancreas    scProto (masked-trained)                    0.670                           0.591                          0.092                       0.685   
            SEACells (same masked graph)                0.556                           0.473                          0.078                       0.537   
            SEACells (scPoli (Stage-1))                 0.274                           0.252                          0.065                       0.261   
            Leiden (scPoli (Stage-1))                   0.386                           0.356                          0.058                       0.385   
            SEACells (scVI)                             0.311                           0.274                          0.066                       0.300   
            Leiden (scVI)                               0.363                           0.341                          0.115                       0.354   
            SEACells (Harmony)                          0.259                           0.236                          0.071                       0.244   
            Leiden (Harmony)                            0.569                           0.528                          0.091                       0.601   
lung        scProto (masked-trained)                    0.725                           0.661                          0.024                       0.736   
            SEACells (same masked graph)                0.598                           0.521                          0.070                       0.579   
            SEACells (scPoli (Stage-1))                 0.324                           0.306                          0.053                       0.313   
            Leiden (scPoli (Stage-1))                   0.529                           0.493                          0.084                       0.536   
            SEACells (scVI)                             0.344                           0.326                          0.070                       0.331   
            Leiden (scVI)                               0.549                           0.503                          0.085                       0.559   
            SEACells (Harmony)                          0.318                           0.281                          0.072                       0.303   
            Leiden (Harmony)                            0.681                           0.606                          0.089                       0.726   
pbmc-immune scProto (masked-trained)                    0.671                           0.621                          0.060                       0.723   
            SEACells (same masked graph)                0.427                           0.394                          0.023                       0.420   
            SEACells (scPoli (Stage-1))                 0.223                           0.213                          0.028                       0.220   
            Leiden (scPoli (Stage-1))                   0.426                           0.449                          0.184                       0.431   
            SEACells (scVI)                             0.210                           0.190                          0.030                       0.209   
            Leiden (scVI)                               0.272                           0.323                          0.208                       0.270   
            SEACells (Harmony)                          0.287                           0.287                          0.067                       0.280   
            Leiden (Harmony)                            0.529                           0.520                          0.115                  

## Not covered here

- **Held-out-*batch*** modularity (E3 in `../experiments_overview.md`) — separate experiment, only valid on datasets with a natural reference/query split.
- **Standard scIB metrics** (E9) — label-based, unaffected by this masking, complementary rather than required here.
- **BBKNN** as a two-step baseline row — excluded from E1 itself (install issue, see `notebook2_pip_install_bug.md`), so nothing to reuse here either.
- **A leak-free SEACells archetype-seeding fix** (`compute_seacells_own_affinity` instead of `compute_seacells_from_affinity`) was considered and deliberately not applied — see the intro cell's rationale.
- **Leiden directly on the masked PCA-arbf graph** was considered and deliberately not added as a row — see the intro cell's rationale (repeats SEACells' batch-effect handicap, doesn't isolate anything new).